# 01 · Organizar y limpiar los datos (exploración)

Primer notebook del proceso de exploración que hizo Mateo antes de integrar hallazgos al pipeline de `src/`.
Carga las tablas de Ford en una base DuckDB local, las alinea en el tiempo y arma una vista de viajes limpios.

**Entradas:** `Datasets/Static`, `Datasets/TripSummary`, `Datasets/Dynamic` · **Salida:** `salidas/fic.duckdb`

Decisiones que se tomaron acá (detalle en `README.md` de esta carpeta):
- Fallados de la v2 (277 vehículos, primer evento de cada uno) + sanos de la versión original (727).
- Dos orígenes de fecha: producción = 31/07/2023 + `ProductionDay` (fallados) y 19/01/2025 + `ProductionDay` (sanos),
  validado contra el primer registro de telemetría de cada vehículo.
- **Todo lo anterior a la venta se descarta**: el 8,3 % de las filas era uso en planta/concesionario (95 % de viajes cortos).
- Los viajes de 0 km con duración se conservan como **ralentí**.

In [ ]:
import os
import duckdb
import pandas as pd

# Rutas relativas a notebooks/exploracion/ (los CSV de Ford quedan en Datasets/, no versionado)
DATOS   = "../../Datasets/"
STATIC  = DATOS + "Static/"
TRIPS   = DATOS + "TripSummary/"
DYN     = DATOS + "Dynamic/"
SALIDAS = "salidas/"          # fic.duckdb, dataset.parquet, etc. (no versionado)
os.makedirs(SALIDAS, exist_ok=True)

# DuckDB siempre con límite de recursos (regla del proyecto: sin límite puede congelar una máquina de 16 GB)
con = duckdb.connect(SALIDAS + "fic.duckdb", config={"memory_limit": "4GB", "threads": 4})
con.sql("SET TimeZone='UTC'")

## Mirar antes de tocar

Columnas y tipos de la tabla estática v2, y vehículos repetidos (21 fallados tienen dos fechas de evento).

In [ ]:
f_static = STATIC + "StaticInformation_FailedVins_v2_20260924_144239.csv"

con.sql(f"select * from read_csv_auto('{f_static}') limit 5").df()

In [4]:
con.sql(f"describe select * from read_csv_auto('{f_static}')").df()

,column_name,column_type,null,key,default,extra
0,VehicleCode,VARCHAR,YES,None,None,None
1,IdentificationDaysSinceProduction,BIGINT,YES,None,None,None
2,daysUntilSale,BIGINT,YES,None,None,None
3,ProductionDay,BIGINT,YES,None,None,None
4,Engine,VARCHAR,YES,None,None,None
5,ModelSeries,VARCHAR,YES,None,None,None
6,SalesCountry_cd,VARCHAR,YES,None,None,None
7,SalesCity,VARCHAR,YES,None,None,None


In [ ]:
con.sql(f"""
select VehicleCode, count(*) as veces
from read_csv_auto('{f_static}')
group by VehicleCode
having count(*) > 1
order by VehicleCode
""").df()

## Tabla de vehículos

Un registro por vehículo con fechas de producción, venta y evento. Los 15 vehículos que figuran en ambas listas quedan como fallados.

In [19]:
con.sql(f"""
create or replace table vehiculos as
with fallados as (
    select VehicleCode, Engine, ModelSeries,
           SalesCountry_cd as pais, SalesCity as ciudad,
           daysUntilSale, 1 as failed,
           date '2023-07-31' + ProductionDay::int as prod_date,
           date '2023-07-31' + (ProductionDay + daysUntilSale)::int as venta_date,
           date '2023-07-31' + (ProductionDay + IdentificationDaysSinceProduction)::int as event_date
    from read_csv_auto('{STATIC}StaticInformation_FailedVins_v2_20260924_144239.csv')
    qualify row_number() over (partition by VehicleCode
                               order by IdentificationDaysSinceProduction) = 1
),
sanos as (
    select VehicleCode, Engine, ModelSeries,
           SalesCountry_cd as pais, SalesCity as ciudad,
           daysUntilSale, 0 as failed,
           date '2025-01-19' + ProductionDay::int as prod_date,
           date '2025-01-19' + (ProductionDay + daysUntilSale)::int as venta_date,
           null::date as event_date
    from read_csv_auto('{STATIC}StaticInformation_NotFailedVins_v2_20260924_144209.csv')
    where VehicleCode not in (select VehicleCode from fallados)
    qualify row_number() over (partition by VehicleCode order by ProductionDay) = 1
)
select * from fallados
union all
select * from sanos
""")
con.sql("select failed, count(*) as vehiculos, count(venta_date) as con_venta from vehiculos group by failed").df()

,failed,vehiculos,con_venta
0,0,727,709
1,1,277,277


## Viajes y telemetría

Cada archivo aporta solo los vehículos de su grupo (si no, los duplicados entre listas mezclan viajes de sanos como fallados). Control: fallados 934.463 viajes / 277 vehículos; sanos 1.546.575 / 727.

In [ ]:
con.sql(f"""
create or replace table trips as
select * from read_csv_auto('{TRIPS}TripSummary_Failed_SelectionVins_vehiclecode_v2 (2).csv')
where VehicleCode in (select VehicleCode from vehiculos where failed = 1)
union all by name
select * from read_csv_auto('{TRIPS}TripSummary_NotFailed_SelectionVins.csv')
where VehicleCode in (select VehicleCode from vehiculos where failed = 0)
""")

con.sql("""
select v.failed, count(*) as viajes, count(distinct t.VehicleCode) as vehiculos
from trips t join vehiculos v using (VehicleCode)
group by v.failed
""").df()

In [11]:
con.sql(f"""
create or replace table dynamic as
select * from read_csv_auto('{DYN}DynamicInformation_Failed_SelectionVins_v2.csv')
where VehicleCode in (select VehicleCode from vehiculos where failed = 1)
union all by name
select * from read_csv_auto('{DYN}DynamicInformation_NotFailed_SelectionVins.csv')
where VehicleCode in (select VehicleCode from vehiculos where failed = 0)
""")

Chequeo de escala: `Acumulation` llega a 100 en ambos grupos (mediana 40, p99 96), así que no hace falta normalizar entre versiones.

In [12]:
con.sql("""
select v.failed,
       count(*)                                   as registros,
       max(d.Acumulation)                         as maximo,
       approx_quantile(d.Acumulation, [0.5, 0.99]) as mediana_y_p99
from dynamic d
join vehiculos v using (VehicleCode)
group by v.failed
""").df()

,failed,registros,maximo,mediana_y_p99
0,0,6245631,100,"[40, 96]"
1,1,3595216,100,"[40, 96]"


## Calidad de datos

Conteo de problemas antes de limpiar (va al informe).

In [13]:
calidad = con.sql("""
select count(*) as total,
  count(*) filter (where OdometerTripEnd = OdometerTripStart)  as cero_km,
  count(*) filter (where OdometerTripEnd < OdometerTripStart)  as odometro_retrocede,
  count(*) filter (where TripDatetimeEnd <= TripDatetimeStart) as duracion_cero,
  count(*) filter (where FuelLvlStartPc > 100)                  as combustible_mayor_100,
  count(*) filter (where KilometerPerHour > 200)                as velocidad_absurda,
  count(*) filter (where AirTemperatureAvg <= -60)              as temp_aire_centinela
from trips
""").df()
calidad

,total,cero_km,odometro_retrocede,duracion_cero,combustible_mayor_100,velocidad_absurda,temp_aire_centinela
0,2481038,843784,262,134179,239409,441,99


## Viajes limpios (desde la venta) y ralentí

In [20]:
con.sql("""
create or replace view trips_limpios as
select t.* exclude (KilometerPerHour, FuelLvlStartPc, FuelLvlEndPc, AirTemperatureAvg),
       case when KilometerPerHour <= 200 then KilometerPerHour end   as KilometerPerHour,
       least(FuelLvlStartPc, 100)                                    as FuelLvlStartPc,
       least(FuelLvlEndPc, 100)                                      as FuelLvlEndPc,
       case when AirTemperatureAvg > -60 then AirTemperatureAvg end  as AirTemperatureAvg,
       OdometerTripEnd - OdometerTripStart                           as km,
       date_diff('minute', TripDatetimeStart, TripDatetimeEnd)       as minutos,
       TripDatetimeStart::date - v.prod_date                         as edad_dias
from trips t
join vehiculos v using (VehicleCode)
where OdometerTripEnd > OdometerTripStart
  and TripDatetimeEnd > TripDatetimeStart
  and TripDatetimeStart::date >= v.venta_date
""")
con.sql("""select v.failed, count(*) as viajes
           from trips_limpios join vehiculos v using (VehicleCode) group by v.failed""").df()

,failed,viajes
0,0,1015482
1,1,596217


In [21]:
con.sql("""
create or replace view ralenti as
select t.VehicleCode, t.TripDatetimeStart::date as dia,
       date_diff('minute', TripDatetimeStart, TripDatetimeEnd) as minutos
from trips t
join vehiculos v using (VehicleCode)
where OdometerTripEnd = OdometerTripStart
  and TripDatetimeEnd > TripDatetimeStart
  and date_diff('minute', TripDatetimeStart, TripDatetimeEnd) <= 600
  and TripDatetimeStart::date >= v.venta_date
""")
con.sql("select count(*) as sesiones, approx_quantile(minutos, [0.5, 0.9, 0.99]) as p50_p90_p99 from ralenti").df()

,sesiones,p50_p90_p99
0,650809,"[1, 6, 30]"


Sesgo de cohorte: los fallados son más viejos que los sanos, por eso edad, fecha y odómetro nunca se usan como features.

In [15]:
con.sql("""
select v.failed, approx_quantile(edad_dias, [0.1, 0.5, 0.9]) as edad_p10_p50_p90
from trips_limpios
join vehiculos v using (VehicleCode)
group by v.failed
""").df()

,failed,edad_p10_p50_p90
0,0,"[117, 266, 443]"
1,1,"[134, 334, 567]"


In [22]:
con.close()
